In [2]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [4]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

### Explore the Data

In [5]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavit

((569, 30),
 (569,),
 None,
 None,
 Diagnosis
 B    357
 M    212
 Name: count, dtype: int64)

#### Check for duplicates

In [6]:
X.duplicated().sum()

np.int64(0)

In [7]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

/tmp/ipykernel_1576/291255995.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({"B": 0, "M": 1})


#### Split the dataset

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((455, 30), (455,), (114, 30), (114,))

#### Select top 10 features using SelectKBest

In [28]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
import pandas as pd

# Create a fresh SelectKBest instance and fit it on the original splits
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train, y_train)

selected_cols = selector.get_feature_names_out(X_train.columns)

X_train = X_train[selected_cols].copy()
X_test = X_test[selected_cols].copy()

print("Selected features:", list(X_train.columns))
print("New X_train shape:", X_train.shape)

Selected features: ['radius1', 'perimeter1', 'area1', 'concavity1', 'concave_points1', 'radius3', 'perimeter3', 'area3', 'concavity3', 'concave_points3']
New X_train shape: (455, 10)


**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

### Define models, apply transformation, and evaluate the models

**Define Candidate Model Pipelines**

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Standardized Cross Validation Evaluation

In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import time
import numpy as np

skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


In [33]:
def evaluate_model(model_name, model, x, y, cv):

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    start_time = time.perf_counter()
    fit_count = 0

    for train_index, val_index in cv.split(X=x, y=y):

        clone_clf = clone(model)

        # Safely index both pandas DataFrames/Series and NumPy arrays
        X_train_folds = x.iloc[train_index] if hasattr(x, 'iloc') else x[train_index]
        y_train_folds = y.iloc[train_index] if hasattr(y, 'iloc') else y.iloc[train_index] if hasattr(y, 'iloc') else np.array(y)[train_index]

        X_val_folds = x.iloc[val_index] if hasattr(x, 'iloc') else x[val_index]
        y_val_folds = y.iloc[val_index] if hasattr(y, 'iloc') else np.array(y)[val_index]

        # ONE fit for this fold
        clone_clf.fit(X_train_folds, y_train_folds)
        fit_count += 1

        # Class predictions
        y_pred = clone_clf.predict(X_val_folds)

        # Continuous scores for AUC
        if hasattr(clone_clf, "predict_proba"):
            y_score = clone_clf.predict_proba(X_val_folds)[:, 1]

        elif hasattr(clone_clf, "decision_function"):
            y_score = clone_clf.decision_function(X_val_folds)

        else:
            raise ValueError(
                f"{model_name} cannot produce scores for AUC"
            )

        # Save this fold's results
        y_true_all.extend(y_val_folds)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

    runtime = time.perf_counter() - start_time

    # Convert to arrays
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_score_all = np.array(y_score_all)

    # Metrics
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    auc = roc_auc_score(y_true_all, y_score_all)

    cm = confusion_matrix(
        y_true=y_true_all,
        y_pred=y_pred_all
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc,
        "Runtime_seconds": runtime,
        "Model_evaluations": 1,
        "CV_fits": fit_count,
        "Confusion_matrix": cm
    }

In [34]:
results = []

results.append(
    evaluate_model( "LR", LR_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "SVC", SVC_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "RF", RF_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "GB", GB_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "XGB", XGB_pipeline, X_train, y_train, skfolds)
)

In [35]:
results_df = pd.DataFrame(results)
results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "AUC",
        "Runtime_seconds",
        "Model_evaluations",
        "CV_fits"
    ]
]

,Model,Accuracy,Precision,Recall,F1,AUC,Runtime_seconds,Model_evaluations,CV_fits
0,LR,0.942857,0.944444,0.900000,0.921687,0.983137,0.317503,1,3
1,SVC,0.945055,0.955975,0.894118,0.924012,0.978411,0.150758,1,3
2,RF,0.940659,0.923077,0.917647,0.920354,0.978277,2.320634,1,3
3,GB,0.940659,0.928144,0.911765,0.919881,0.974211,2.472756,1,3
4,XGB,0.949451,0.940120,0.923529,0.931751,0.978720,1.075439,1,3
